# 05-09 Binding: 모델에 파라미터와 도구 미리 연결하기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `bind(stop=...)`로 특정 토큰에서 생성을 중단시켜 출력 길이를 제어하는 방법을 구현할 수 있어요
- `bind(tools=[...])`로 OpenAI 함수(Function)와 도구(Tool) 스키마를 모델에 연결하고, 구조화된 응답을 얻는 방법을 설명할 수 있어요
- `bind()`로 고정된 파라미터가 체인 전체에 전파되는 원리를 이해할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- OpenAI API의 Function Calling 개념
- LCEL 파이프 연산자(`|`) 사용법
- JSON 스키마(Schema) 기초

---

`Runnable.bind()`를 사용하면 체인 실행 시 상수 인자를 전달할 수 있어요.

**주요 활용:**
- Stop 토큰(Stop Token) 설정
- OpenAI Functions 연결
- OpenAI Tools 연결
- 기타 모델 파라미터 고정

이를 통해 체인 구성 시점에 파라미터를 미리 설정하여 매번 `invoke()` 시 반복적으로 전달할 필요가 없어요.

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)


## 1. Stop 토큰 바인딩

`bind(stop=...)`를 사용하여 특정 토큰에서 생성이 중단되도록 설정할 수 있어요.

출력 형식이 정해진 경우, 특정 섹션까지만 필요하다면 stop 토큰으로 불필요한 토큰 소비를 줄일 수 있어요.

> **실무 팁:** stop 토큰은 배포 환경에서 출력 길이를 엄격하게 제어해야 할 때 유용해요. 예를 들어 "SOLUTION:" 이전의 문제 정의 부분만 필요한 경우 불필요한 LLM 호출 비용을 절감할 수 있어요.

In [ ]:
# ============================================================
# TODO: model.bind(stop="SOLUTION")으로 stop 토큰을 바인딩하고 기본 체인과 비교하세요
# 힌트: model.bind(stop="SOLUTION") — 이 문자열이 나타나면 생성 중단
# 예상 결과: stop 토큰 바인딩 체인은 "EQUATION: ..." 부분만 출력
# ============================================================

# ---------------------------------------------------
# bind(stop=...): stop 토큰으로 생성 길이 제어
# ---------------------------------------------------

# 1단계: 프롬프트 정의
# EQUATION:, SOLUTION: 형식으로 출력하도록 지시
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "대수 기호를 사용하여 다음 방정식을 작성한 다음 풀이하세요. "
            "다음 형식을 사용하세요:\n\nEQUATION:...\nSOLUTION:...\n\n",
        ),
        ("human", "{equation_statement}"),
    ]
)

# 2단계: 기본 체인 (stop 토큰 없음 — 전체 출력)


In [ ]:
# Stop 토큰을 바인딩한 체인
# 
# "SOLUTION" 토큰에서 생성이 중단됨


## 2. OpenAI 함수 연결

`bind()`에 OpenAI 함수(Function) 스키마를 전달하면, 모델이 필요에 따라 함수를 호출해 구조화된 답을 반환할 수 있어요.

필요 시 `tool_choice`를 추가로 설정해 특정 함수를 강제로 호출할 수도 있어요.

> **주의:** `tool_choice`를 사용하지 않으면 모델이 함수를 호출할지 여부를 스스로 결정해요. 응답을 반드시 구조화된 형태로 받고 싶을 때는 `tool_choice={"type": "function", "function": {"name": "함수명"}}`을 추가하세요.

In [ ]:
# ============================================================
# TODO: model.bind(tools=[solver_tool])로 OpenAI 함수를 바인딩하고 구조화된 응답을 받으세요
# 힌트: tools=[{"type": "function", "function": {...}}] 형식으로 정의
# 예상 결과: AIMessage.tool_calls에 {"name": "solver", "args": {...}} 포함
# ============================================================

# ---------------------------------------------------
# bind(tools=[...]): OpenAI Function Calling — 구조화된 응답 유도
# ---------------------------------------------------


In [ ]:
# Functions 바인딩 체인 실행


## 3. OpenAI Tools 바인딩

`bind(tools=[...])`에 도구(Tool) 목록을 전달하면 모델이 필요한 경우 해당 도구를 자동으로 호출해요.

Function Calling과 달리, Tools API는 한 번의 호출로 여러 도구를 병렬로 호출할 수 있어요.

In [ ]:
# ============================================================
# TODO: bind(tools=[...])로 날씨 조회 도구를 모델에 바인딩하세요
# 힌트: tools 리스트에 {"type": "function", "function": {...}} 형식으로 정의
# 예상 결과: AIMessage.tool_calls에 도시별 날씨 조회 함수 호출 정보 포함
# ============================================================

# ---------------------------------------------------
# bind(tools=[...]): Tools API — 여러 도구를 병렬 호출 가능
# ---------------------------------------------------


In [ ]:
# Tools 바인딩 모델 실행


## 마무리 요약

### bind() 메서드의 주요 활용 비교

| 활용 | 코드 예시 | 효과 |
|------|-----------|------|
| Stop 토큰 | `model.bind(stop="SOLUTION")` | 특정 토큰에서 생성 중단 |
| Function Calling | `model.bind(tools=[fn_schema])` | 구조화된 함수 호출 응답 |
| Tools API | `model.bind(tools=[tool_schema])` | 다중 도구 병렬 호출 가능 |

### 핵심 요점

- `bind()`는 체인 구성 시점에 파라미터를 고정하여 실행마다 반복 전달하는 번거로움을 없애줘요
- Stop 토큰으로 출력 길이를 정밀하게 제어하고 불필요한 토큰 비용을 절감할 수 있어요
- `bind(tools=[...])`는 모델이 필요 시 도구를 자동으로 선택하고 호출해요
- `tool_choice`를 추가하면 특정 함수 호출을 강제할 수 있어요

### 다음 노트북 예고

다음 노트북(`10-Fallbacks.ipynb`)에서는 기본 모델이나 체인이 실패했을 때 자동으로 대체 경로로 전환하는 폴백(Fallback) 패턴을 배워요. 안정적인 LLM 서비스를 구축하기 위한 필수 패턴이에요.